# Fine-tune YOLO cho vật thể riêng (vision-count)

Notebook này train lại model YOLO để nhận ra **vật thể của bạn** (ốc vít, viên thuốc, cá giống...).

**Chuẩn bị trước** (xem `training/README.md` trong project):
1. Ảnh đã gán nhãn theo định dạng YOLO, xếp đúng cấu trúc `dataset/images/{train,val}` và `dataset/labels/{train,val}`, kèm `dataset/data.yaml`.
2. Đã chạy `python training/check_dataset.py dataset` trên máy và **không có lỗi**.
3. Nén thư mục `dataset/` thành file **`dataset.zip`**.

**Bật GPU miễn phí:** menu **Runtime → Change runtime type → T4 GPU → Save**. Sau đó chạy lần lượt từng ô (bấm ▶ ở mỗi ô, hoặc Shift + Enter).

## 1. Kiểm tra đã có GPU chưa
Phải thấy bảng có chữ **Tesla T4**. Nếu báo lỗi `command not found`: chưa bật GPU, xem hướng dẫn ở trên.

In [ ]:
!nvidia-smi

## 2. Cài Ultralytics

In [ ]:
!pip install -q ultralytics

## 3. Tải bộ dữ liệu lên
Chạy ô dưới, bấm **Choose Files** và chọn `dataset.zip`. Sau khi giải nén phải có `/content/dataset/data.yaml`.

In [ ]:
from google.colab import files

uploaded = files.upload()  # chọn dataset.zip
!unzip -q -o dataset.zip -d /content/

## 4. Kiểm tra nhanh bộ dữ liệu

In [ ]:
from pathlib import Path

root = Path("/content/dataset")
assert (root / "data.yaml").is_file(), "Không thấy /content/dataset/data.yaml: kiểm tra lại file zip (bên trong phải là thư mục dataset/)"
for split in ("train", "val"):
    n_img = len(list((root / "images" / split).glob("*")))
    n_lbl = len(list((root / "labels" / split).glob("*.txt")))
    print(f"{split}: {n_img} ảnh, {n_lbl} file nhãn")
print(open(root / "data.yaml", encoding="utf-8").read())

## 5. Train

- Bắt đầu từ model **pretrained** `yolo11n.pt` (transfer learning): học nhanh hơn và cần ít ảnh hơn nhiều so với train từ đầu.
- `epochs=100`: số vòng học. `patience=20`: tự dừng sớm nếu 20 vòng liền không tiến bộ.
- Với vài trăm ảnh, trên GPU T4 thường mất **15-40 phút**. Đừng đóng tab trong lúc chạy.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")
results = model.train(data="/content/dataset/data.yaml", epochs=100, imgsz=640, patience=20)

## 6. Đánh giá

- **mAP50**: độ chính xác khi khung dự đoán trùng khung thật từ 50% trở lên. **Trên 0.8 là tốt**; dưới 0.5 thì nên thêm ảnh hoặc sửa nhãn.
- **mAP50-95**: chặt hơn (đòi khung khớp sát), thường thấp hơn mAP50.

In [ ]:
metrics = model.val()
print(f"mAP50    = {metrics.box.map50:.3f}")
print(f"mAP50-95 = {metrics.box.map:.3f}")

## 7. Thử dự đoán một ảnh trong tập val

In [ ]:
from IPython.display import Image as Show

sample = next((root / "images" / "val").glob("*"))
pred = model.predict(sample, conf=0.25, save=True)
print("Đếm được:", len(pred[0].boxes), "vật")
Show(filename=str(Path(pred[0].save_dir) / sample.name))

## 8. Tải model đã train về máy
File `best.pt` là model tốt nhất trong quá trình train.

In [ ]:
best = Path(model.trainer.best)  # thường là /content/runs/detect/train/weights/best.pt
print(best)
files.download(str(best))

## 9. Dùng model mới trong vision-count

1. Chép `best.pt` vào thư mục `models/` của project.
2. Mở `vision_count/config.py`, thêm một dòng vào `AVAILABLE_MODELS`:
   ```python
   "cua_toi": ("Model của tôi", "best.pt"),
   ```
3. Chạy lại `python app.py`, chọn **Model của tôi** trong ô **Model**. Ô **Chỉ đếm các loại** sẽ tự đổi sang các loại trong `names` của `data.yaml`.